# FastPower (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_cpp.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `FastPower`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/FastPower_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono variabili globali, e la traccia si attiva con `trace = true`.

## Elevamento a potenza veloce: FastPower

**Input:** un numero $x$ e un numero naturale $n$. **Output:** $x^n$, per quadrati ripetuti. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor + 2$ chiamate, al più $2(\lfloor\log_2 n\rfloor + 1)$ moltiplicazioni.

Pseudocodice: sezione 14.7 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [1]:
%%writefile fast_power.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
long long calls = 0;    // numero di chiamate = nodi dell'albero delle chiamate
int maxDepth = 0;       // profondita' della ricorsione = altezza massima dello stack
long long mults = 0;    // moltiplicazioni (FastPower)
long long moves = 0;    // mosse (Hanoi)
int ev = 0;             // contatore degli eventi della traccia
int stk[1000];          // lo stack delle chiamate: gli argomenti delle chiamate attive (al massimo 1000)
int sp = 0;             // numero di chiamate attive
long long seen[64];     // seen[x] = numero di chiamate con argomento x
bool trace = false;     // true: stampa la traccia dell'esecuzione

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Riga | Esecuzioni\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Passi totali S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // come sprintf, ma restituisce una stringa
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // la chiamata termina: il suo frame esce dallo stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // riga numerata della traccia, con lo stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

int FloorLog2(int n)                               // floor(log2 n) con un ciclo (solo per la formula dei passi)
{
    int k = 0;
    while (n > 1)
    {
        n = n / 2;
        k = k + 1;
    }
    return k;
}

// FastPower(x, n)   i valori troppo grandi traboccano in C++ e Java, non in Python
long long FastPower(long long x, int n)
{
    Push(n); cnt[2]++;
    if (n == 0)
    {
        cnt[3]++;
        if (trace) Say(Str("FastPower(%lld, 0)", x), "caso base, restituisce 1");
        Pop();
        return 1;
    }
    if (trace) Say(Str("FastPower(%lld, %d)", x, n), Str("%d != 0 => chiama FastPower(%lld, %d)   (%d/2 = %d)", n, x, n / 2, n, n / 2));
    long long y = FastPower(x, n / 2); cnt[5]++;   // una sola chiamata ricorsiva: il risultato e' in y
    cnt[6]++;
    long long r;
    if (n % 2 == 0)
    {
        cnt[7]++;
        mults += 1;
        r = y * y;
        if (trace) Say(Str("FastPower(%lld, %d)", x, n), Str("y = %lld, %d e' pari => restituisce %lld * %lld = %lld", y, n, y, y, r));
    }
    else
    {
        cnt[9]++;
        mults += 2;
        r = x * y * y;
        if (trace) Say(Str("FastPower(%lld, %d)", x, n), Str("y = %lld, %d e' dispari => restituisce %lld * %lld * %lld = %lld", y, n, x, y, y, r));
    }
    Pop();
    return r;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Esempio delle dispense: FastPower(3, 13)
    long long x = 3;
    int n = 13;
    Reset();
    trace = true;
    printf("%-28s| %s\n", "chiamata", "azione");
    long long r = FastPower(x, n);
    trace = false;
    printf("%lld^%d = %lld, chiamate = %lld, moltiplicazioni = %lld\n", x, n, r, calls, mults);
    PrintSteps({2, 3, 5, 6, 7, 9});
    printf("S(n) = 4 floor(log2 n) + 6 = %d\n", 4 * FloorLog2(n) + 6);

    // Quante moltiplicazioni: circa 2 log2 n invece di n (x = 1, per non traboccare)
    printf("\n");
    printf("      n | chiamate | moltiplicazioni | S(n) | 4 floor(log2 n) + 6\n");
    int values[] = {1, 2, 13, 100, 1000, 1000000};
    for (int v : values)
    {
        Reset();
        FastPower(1, v);
        printf("%7d | %8lld | %15lld | %4lld | %d\n", v, calls, mults, Total(), 4 * FloorLog2(v) + 6);
    }
    return 0;
}
#endif

Writing fast_power.cpp


In [2]:
!g++ -std=c++17 -O2 -o fast_power fast_power.cpp && ./fast_power

chiamata                    | azione
FastPower(3, 13)            | 13 != 0 => chiama FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => chiama FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => chiama FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => chiama FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | caso base, restituisce 1
      FastPower(3, 1)       | y = 1, 1 e' dispari => restituisce 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 e' dispari => restituisce 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 e' pari => restituisce 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 e' dispari => restituisce 3 * 729 * 729 = 1594323
3^13 = 1594323, chiamate = 5, moltiplicazioni = 7
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Passi totali S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n | chiamate | moltiplicazioni | S(n) | 4 f

**Prova tu.** Cambia $x$ e $n$ e conta le moltiplicazioni.

In [3]:
%%writefile prova_fast_power.cpp
#define SOLO_LIBRERIA   // non eseguire il main di fast_power.cpp
#include "fast_power.cpp"

int main()
{
    long long x = 2;
    int n = 10;        // <-- cambia x e n (con x e n grandi C++ e Java traboccano)
    Reset();
    trace = true;
    printf("%-28s| %s\n", "chiamata", "azione");
    long long r = FastPower(x, n);
    trace = false;
    printf("%lld^%d = %lld, moltiplicazioni = %lld\n", x, n, r, mults);
    return 0;
}

Writing prova_fast_power.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_fast_power prova_fast_power.cpp && ./prova_fast_power

chiamata                    | azione
FastPower(2, 10)            | 10 != 0 => chiama FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => chiama FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => chiama FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => chiama FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | caso base, restituisce 1
      FastPower(2, 1)       | y = 1, 1 e' dispari => restituisce 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 e' pari => restituisce 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 e' dispari => restituisce 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 e' pari => restituisce 32 * 32 = 1024
2^10 = 1024, moltiplicazioni = 6


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).